## **Importing necessary libraries and tools along with finding the kaggle directory paths.**

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## **Taking a look at the data**

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')
train.describe()

In [ ]:
X = train.drop(labels=["label"], axis=1) / 255.0 
y = train["label"] # The test dataset doesn't have a label column
# Reshaping is necessary for the Data Augmentation part
X_reshaped = X.values.reshape(-1, 28, 28, 1)
X_train, X_val, y_train, y_val = train_test_split(X_reshaped, y, test_size=0.1, random_state=55)

## **Data Augmentation and Model Builidng**

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,  
    zoom_range=0.1,     
    width_shift_range=0.1, 
    height_shift_range=0.1
)
datagen.fit(X_train)
model = Sequential([
    Input(shape=(28, 28, 1)), # OG code used to be Flatten(input_shape=(28, 28, 1)) but in newer versions of Tensorflow, a cleaner version exists
    Flatten(),
    Dense(units=512, activation='relu', kernel_regularizer=L2(0.001)),
    Dense(units=256, activation='relu', kernel_regularizer=L2(0.001)),
    Dense(units=10, activation='linear')])

model.compile(optimizer=Adam(learning_rate=1e-3), 
              loss=SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history = model.fit(datagen.flow(X_train, y_train, batch_size=64),
                    epochs=35, 
                    validation_data=(X_val, y_val))

## **Plotting a confusion matrix to see where exactly is the model going wrong**

In [ ]:
val_logits = model.predict(X_val)
val_preds = np.argmax(val_logits, axis=1)
cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Digit Recognizer')
plt.show()

## **We observe that the model is getting confused mainly in the 3/8 and 2/7 numbers**

In [ ]:
def plot_specific_errors(true_label, pred_label, X_val, y_val, val_preds):
    error_indices = np.where((y_val == true_label) & (val_preds == pred_label))[0]
    if len(error_indices) == 0:
        print(f"No errors found where True={true_label} and Pred={pred_label}")
        return
    plt.figure(figsize=(10, 4))
    for i, idx in enumerate(error_indices[:5]): # Show first 5 examples
        plt.subplot(1, 5, i + 1)
        # Reshape the flattened array back to 28x28 for display
        plt.imshow(X_val[idx].reshape(28, 28), cmap='gray')
        plt.title(f"T:{true_label} P:{pred_label}")
        plt.axis('off')
    plt.show()

print("Investigating 8 vs 3 confusion:")
plot_specific_errors(8, 3, X_val, y_val.values, val_preds)

print("Investigating 2 vs 7 confusion:")
plot_specific_errors(2, 7, X_val, y_val.values, val_preds)

## **Submission**

In [ ]:
test = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')
X_test = (test / 255.0).values.reshape(-1, 28, 28, 1)
logits_test = model.predict(X_test)
test_predictions = np.argmax(logits_test, axis=1)
image_ids = np.arange(1, len(test_predictions) + 1)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(test_predictions) + 1),
    "Label": test_predictions
})

submission.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")